05_extract_wavelet_features

# 05 — Extract Wavelet + Additional Forensic Features
**Important:** benchmark first. One image load + one DWT per image. The complete feature set is written once, so later L1 selection does not touch images.

In [1]:
from pathlib import Path
import sys
ROOT=Path.cwd()
while ROOT!=ROOT.parent and not (ROOT/"data").exists(): ROOT=ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
print("PROJECT_ROOT:",ROOT)
DATASET=ROOT/"data"/"raw"/"deepfake_merged_dataset"
print("DATASET:",DATASET)
assert DATASET.exists(), f"Missing dataset: {DATASET}"
import time
from src.features import extract_all,ALL_COLS
paths=list((DATASET/"train/real").iterdir())[:10]+list((DATASET/"train/fake").iterdir())[:10]
t=time.perf_counter()
for p in paths: extract_all(p)
sec=time.perf_counter()-t
print(f"Benchmark: {len(paths)} images in {sec:.2f}s")
print(f"Estimated train+val (304154 minus test): {sec/len(paths)*(230361+48289)/3600:.2f} hours")
print("STOP here if the estimate is too slow.")


PROJECT_ROOT: d:\deepfake_noise_wavelet_ml
DATASET: d:\deepfake_noise_wavelet_ml\data\raw\deepfake_merged_dataset
Benchmark: 20 images in 1.21s
Estimated train+val (304154 minus test): 4.69 hours
STOP here if the estimate is too slow.


In [2]:
# Run ONLY after accepting the benchmark.
import csv
from tqdm import tqdm
from src.features import extract_all,ALL_COLS
def run(split):
    paths=[(p,0) for p in (DATASET/split/"real").iterdir() if p.is_file()]+[(p,1) for p in (DATASET/split/"fake").iterdir() if p.is_file()]
    paths.sort(key=lambda x:str(x[0])); out=ROOT/"data/processed"/f"{split}_all_forensic_features.csv"
    with out.open("w",newline="",encoding="utf-8") as f:
        w=csv.writer(f); w.writerow(ALL_COLS+["filepath","label"])
        for p,y in tqdm(paths,desc=split):
            d=extract_all(p); w.writerow([d[c] for c in ALL_COLS]+[str(p),y])
for s in ["train","val"]: run(s)


val: 100%|██████████| 48289/48289 [51:25<00:00, 15.65it/s]
